# Hush Ollama Experiment

This notebook runs an **ephemeral, local Ollama experiment** in a Google Colab runtime. It does not create a production inference service and it does not expose Ollama with a tunnel or proxy. Colab `localhost` is the Colab VM, not your laptop and not a Vercel function.

Before running: select **Runtime → Change runtime type → GPU** when a GPU is available. Run cells top to bottom. Colab runtimes and downloaded model files can disappear when the session ends.

In [ ]:
import platform, sys, os
print('Python:', sys.version)
print('Platform:', platform.platform())
print('Working directory:', os.getcwd())

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], text=True, capture_output=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('No NVIDIA GPU detected. CPU inference may be slow or unsuitable for qwen3:4b.')
    print(result.stderr.strip())

In [ ]:
import subprocess
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'curl', 'ca-certificates', 'zstd'], check=True)
print('OS utilities installed: curl, ca-certificates, zstd.')

In [ ]:
import subprocess
# Official installer; review its output in your runtime before continuing.
subprocess.run(['bash', '-c', 'curl -fsSL https://ollama.com/install.sh | sh'], check=True)
print('Ollama installation completed.')

In [ ]:
import subprocess, time
ollama_process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(2)
print('Started ollama serve with PID', ollama_process.pid)
# Ollama is deliberately kept private to this Colab runtime; do not create a tunnel or proxy.

In [ ]:
import time, requests
base_url = 'http://127.0.0.1:11434'
for attempt in range(30):
    try:
        response = requests.get(base_url + '/api/tags', timeout=3)
        response.raise_for_status()
        print('Ollama API is ready:', response.json())
        break
    except requests.RequestException as error:
        if attempt == 29:
            raise RuntimeError('Ollama did not become ready. Re-run the serve cell and inspect its output.') from error
        time.sleep(1)

## Download model (intentional action)
Run the next cell only when you want to download `qwen3:4b`. It can require significant disk, RAM, VRAM, and time. If it fails, choose a smaller model or a larger runtime; do not assume the model exists.

In [ ]:
import subprocess
subprocess.run(['ollama', 'pull', 'qwen3:4b'], check=True)
print('qwen3:4b download completed.')

In [ ]:
import subprocess
subprocess.run(['ollama', 'list'], check=True)

In [ ]:
response = requests.get(base_url + '/api/tags', timeout=10)
response.raise_for_status()
print(response.json())

In [ ]:
payload = {
    'model': 'qwen3:4b',
    'messages': [{'role': 'user', 'content': 'Reply with exactly: Hush local inference works.'}],
    'stream': False,
}
response = requests.post(base_url + '/api/chat', json=payload, timeout=180)
if response.status_code != 200:
    print('Model request failed:', response.status_code, response.text)
else:
    print('Successful inference:', response.json().get('message', {}).get('content', ''))

In [ ]:
stream_payload = {
    'model': 'qwen3:4b',
    'messages': [{'role': 'user', 'content': 'Count from one to three, one number per token.'}],
    'stream': True,
}
with requests.post(base_url + '/api/chat', json=stream_payload, stream=True, timeout=180) as response:
    response.raise_for_status()
    for line in response.iter_lines(decode_unicode=True):
        if not line:
            continue
        event = __import__('json').loads(line)
        print(event.get('message', {}).get('content', ''), end='', flush=True)
        if event.get('done'):
            print('\nStreaming completed.')
            break

## Troubleshooting and limits

- **Model missing:** run the deliberate `ollama pull qwen3:4b` cell, then rerun `/api/tags`.
- **Ollama unavailable:** rerun the install and `ollama serve` cells; inspect process output and available disk space.
- **Out of memory / slow output:** select a GPU, restart the runtime, reduce concurrent work, or use a smaller model.
- **Successful output:** the non-streaming test prints a message and the streaming test emits text incrementally.
- **Production boundary:** this notebook is a model experiment only. Managed Colab runtimes may end at any time. Do not use tunnels/proxies to publish it, and do not configure its localhost as a Vercel endpoint.